In [1]:
!pip install pandas numpy scikit-learn tensorflow statsmodels matplotlib


 CODE

In [ ]:
import numpy as np
import pandas as pd
import os
import joblib
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

class VegetablePricePredictionModel:
    def __init__(self, data_path):
        # Load and preprocess data
        self.data = pd.read_csv(data_path, header=None)  # Load without header
        self.data.columns = [
            "Date", "Vegetable", "Price_per_kg", "Season", "Temp", "Rainfall", "Humidity",
            "Disaster", "Demand", "Freshness", "Inflation", "Yield_per_Hectare",
            "Holiday_Festival", "COVID_Affected", "Year"
        ]
        self.data['Date'] = pd.to_datetime(self.data['Date'], format='%d-%m-%Y', errors='coerce')
        self.data = self.data.dropna(subset=['Date'])

        # Preprocessing
        self.categorical_columns = ['Disaster', 'Season', 'Holiday_Festival', 'Vegetable']
        self.data = pd.get_dummies(self.data, columns=self.categorical_columns, drop_first=True)

        # Prepare features and target
        self.features_columns = [
            'Temp', 'Rainfall', 'Humidity', 'Demand',
            'Freshness', 'Inflation', 'Yield_per_Hectare'
        ] + [col for col in self.data.columns if any(cat in col for cat in self.categorical_columns)]

        self.X = self.data[self.features_columns]
        self.y = self.data['Price_per_kg']

    def prepare_data(self, test_size=0.2, random_state=42):
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            self.X, self.y, test_size=test_size, random_state=random_state
        )

        # Scale features
        self.feature_scaler = StandardScaler()
        X_train_scaled = self.feature_scaler.fit_transform(X_train)
        X_test_scaled = self.feature_scaler.transform(X_test)

        # Scale target
        self.target_scaler = MinMaxScaler()
        y_train_scaled = self.target_scaler.fit_transform(y_train.values.reshape(-1, 1))
        y_test_scaled = self.target_scaler.transform(y_test.values.reshape(-1, 1))

        return X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled, X_train, X_test, y_train, y_test

    def build_models(self, input_shape):
        # LSTM Model
        lstm_model = Sequential([
            LSTM(64, input_shape=input_shape, return_sequences=True),
            Dropout(0.2),
            LSTM(32),
            Dropout(0.2),
            Dense(16, activation='relu'),
            Dense(1)
        ])
        lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

        # Dense Model for Comparison
        dense_model = Sequential([
            Dense(64, activation='relu', input_dim=input_shape[1]),
            Dropout(0.3),
            Dense(32, activation='relu'),
            Dense(16, activation='relu'),
            Dense(1)
        ])
        dense_model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

        return lstm_model, dense_model

    def train_models(self):
        # Prepare data
        X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled, X_train, X_test, y_train, y_test = self.prepare_data()

        # LSTM requires 3D input
        X_train_lstm = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
        X_test_lstm = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

        # Build models
        lstm_model, dense_model = self.build_models((1, X_train_scaled.shape[1]))

        # Early stopping
        early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

        # Train LSTM
        lstm_history = lstm_model.fit(
            X_train_lstm, y_train_scaled,
            epochs=100,
            batch_size=32,
            validation_split=0.2,
            callbacks=[early_stopping],
            verbose=1
        )

        # Train Dense
        dense_history = dense_model.fit(
            X_train_scaled, y_train_scaled,
            epochs=100,
            batch_size=32,
            validation_split=0.2,
            callbacks=[early_stopping],
            verbose=1
        )

        # Predictions
        lstm_pred_scaled = lstm_model.predict(X_test_lstm)
        dense_pred_scaled = dense_model.predict(X_test_scaled)

        # Inverse transform predictions
        lstm_pred = self.target_scaler.inverse_transform(lstm_pred_scaled).flatten()
        dense_pred = self.target_scaler.inverse_transform(dense_pred_scaled).flatten()

        # Calculate metrics
        metrics = {
            'LSTM': self.calculate_regression_metrics(y_test, lstm_pred),
            'Dense': self.calculate_regression_metrics(y_test, dense_pred)
        }

        # Save models and scalers
        os.makedirs('models', exist_ok=True)
        lstm_model.save('models/lstm_vegetable_price_model.h5')
        dense_model.save('models/dense_vegetable_price_model.h5')

        joblib.dump(self.feature_scaler, 'models/feature_scaler.pkl')
        joblib.dump(self.target_scaler, 'models/target_scaler.pkl')

        # Plot results
        plt.figure(figsize=(12, 6))
        plt.plot(y_test, label='Actual Prices', color='blue')
        plt.plot(lstm_pred, label='LSTM Predictions', color='red', linestyle='--')
        plt.plot(dense_pred, label='Dense Predictions', color='green', linestyle=':')
        plt.title('Vegetable Price Predictions')
        plt.xlabel('Sample Index')
        plt.ylabel('Price per kg')
        plt.legend()
        plt.tight_layout()
        plt.savefig('models/prediction_comparison.png')
        plt.close()

        return metrics, lstm_model, dense_model

    def calculate_regression_metrics(self, y_true, y_pred):
        return {
            'MSE': mean_squared_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'MAE': mean_absolute_error(y_true, y_pred),
            'MAPE': mean_absolute_percentage_error(y_true, y_pred) * 100,
            'R2': r2_score(y_true, y_pred)
        }

    def preprocess_input(self, input_data):
        try:
            # Create DataFrame from input_data
            input_df = pd.DataFrame([input_data])

            # Get all possible categorical columns from training data
            all_cat_columns = [col for col in self.X.columns if any(cat in col for cat in self.categorical_columns)]

            # Initialize all categorical columns with 0
            for col in all_cat_columns:
                if col not in input_df.columns:
                    input_df[col] = 0

            # Ensure input DataFrame has the same columns as training data
            input_df = input_df[self.features_columns]

            # Scale features
            input_scaled = self.feature_scaler.transform(input_df)

            return input_scaled, input_df

        except Exception as e:
            raise ValueError(f"Error preprocessing input data: {str(e)}")

    def predict(self, input_data, lstm_model=None, dense_model=None):
        try:
            # Preprocess input
            input_scaled, input_df = self.preprocess_input(input_data)

            # Load models if not provided
            if lstm_model is None or dense_model is None:
                lstm_model = tf.keras.models.load_model('models/lstm_vegetable_price_model.h5')
                dense_model = tf.keras.models.load_model('models/dense_vegetable_price_model.h5')

            # Reshape input for LSTM (add time step dimension)
            lstm_input = input_scaled.reshape((input_scaled.shape[0], 1, input_scaled.shape[1]))

            # Make predictions
            lstm_prediction_scaled = lstm_model.predict(lstm_input, verbose=0)
            dense_prediction_scaled = dense_model.predict(input_scaled, verbose=0)

            # Inverse transform predictions
            lstm_prediction = self.target_scaler.inverse_transform(lstm_prediction_scaled).flatten()
            dense_prediction = self.target_scaler.inverse_transform(dense_prediction_scaled).flatten()

            # Calculate ensemble prediction (average of both models)
            ensemble_prediction = (lstm_prediction + dense_prediction) / 2

            return {
                'LSTM': float(lstm_prediction[0]),
                'Dense': float(dense_prediction[0]),
                'Ensemble': float(ensemble_prediction[0])
            }

        except Exception as e:
            raise ValueError(f"Error making predictions: {str(e)}")

if __name__ == "__main__":
    # Create an instance of the model
    model_instance = VegetablePricePredictionModel(data_path='main.csv')

    # Train the models
    metrics, lstm_model, dense_model = model_instance.train_models()
    print("Training metrics:", metrics)

    # Sample input data for prediction
    input_data = {
        "Temp": 30,
        "Rainfall": 100,
        "Humidity": 70,
        "Demand": 200,
        "Freshness": 0.8,
        "Inflation": 1.02,
        "Yield_per_Hectare": 4.5,
        "Disaster_Flood": 1,
        "Season_Kharif": 1,
        "Holiday_Festival_None": 1,
        "Vegetable_Cabbage": 1
    }

    # Make predictions using the trained models
    predictions = model_instance.predict(input_data, lstm_model, dense_model)
    print("\nPredictions for the input data:")
    for model, price in predictions.items():
        print(f"{model}: ₹{price:.2f} per kg")

Epoch 1/100


/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1228/1228 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - loss: 0.0120 - val_loss: 0.0047
Epoch 2/100
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.0050 - val_loss: 0.0046
Epoch 3/100
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - loss: 0.0048 - val_loss: 0.0046
Epoch 4/100
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - loss: 0.0047 - val_loss: 0.0046
Epoch 5/100
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.0047 - val_loss: 0.0046
Epoch 6/100
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.0046 - val_loss: 0.0046
Epoch 7/100
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - loss: 0.0046 - val_loss: 0.0046
Epoch 8/100
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0046 - val_loss: 0.0046
Epoch 9/100
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.0046 - val_loss: 0.0046
Epoch 10/100
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - loss: 0.0046 - val_loss: 0.0046
Epoch 11/100
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.0046 - val_loss: 0.0046
Epoch 12/100
1228/1228 ━━

Training metrics: {'LSTM': {'MSE': 69.82421532646691, 'RMSE': 8.356088518347978, 'MAE': 5.715758724899503, 'MAPE': 15.319202053467412, 'R2': 0.835420548915863}, 'Dense': {'MSE': 94.8733476702141, 'RMSE': 9.740295050470191, 'MAE': 6.538841945145639, 'MAPE': 17.19158201436452, 'R2': 0.7763783931732178}}

Predictions for the input data:
LSTM: ₹45.06 per kg
Dense: ₹57.91 per kg
Ensemble: ₹51.49 per kg


In [38]:
# app.py
from flask import Flask, render_template, request, jsonify
import tensorflow as tf
import joblib
import numpy as np
import os
import pandas as pd

app = Flask(__name__)

# Load the models and scalers
def load_models():
    try:
        lstm_model = tf.keras.models.load_model('models/lstm_vegetable_price_model.h5')
        dense_model = tf.keras.models.load_model('models/dense_vegetable_price_model.h5')
        feature_scaler = joblib.load('models/feature_scaler.pkl')
        target_scaler = joblib.load('models/target_scaler.pkl')
        return lstm_model, dense_model, feature_scaler, target_scaler
    except Exception as e:
        print(f"Error loading models: {str(e)}")
        return None, None, None, None

lstm_model, dense_model, feature_scaler, target_scaler = load_models()

def preprocess_input(input_data, feature_scaler):
    # Initialize all possible columns with zeros
    all_columns = [
        'Temp', 'Rainfall', 'Humidity', 'Demand', 'Freshness', 'Inflation',
        'Yield_per_Hectare', 'Disaster_Drought', 'Disaster_Flood', 'Disaster_None',
        'Season_Kharif', 'Season_Rabi', 'Season_Summer', 'Season_Winter',
        'Holiday_Festival_Diwali', 'Holiday_Festival_None', 'Holiday_Festival_Other',
        'Vegetable_Brinjal', 'Vegetable_Cabbage', 'Vegetable_Carrot',
        'Vegetable_Cauliflower', 'Vegetable_Cucumber', 'Vegetable_Ladyfinger',
        'Vegetable_Onion', 'Vegetable_Potato', 'Vegetable_Tomato'
    ]

    input_dict = {col: 0 for col in all_columns}

    # Update with provided values
    for key, value in input_data.items():
        if key in input_dict:
            input_dict[key] = float(value)
        elif key + '_' + value in input_dict:  # For categorical variables
            input_dict[key + '_' + value] = 1

    # Convert to DataFrame
    input_df = pd.DataFrame([input_dict])

    # Scale features
    input_scaled = feature_scaler.transform(input_df)

    return input_scaled

def make_prediction(input_data):
    try:
        # Preprocess input
        input_scaled = preprocess_input(input_data, feature_scaler)

        # Reshape input for LSTM
        lstm_input = input_scaled.reshape((input_scaled.shape[0], 1, input_scaled.shape[1]))

        # Make predictions
        lstm_prediction_scaled = lstm_model.predict(lstm_input, verbose=0)
        dense_prediction_scaled = dense_model.predict(input_scaled, verbose=0)

        # Inverse transform predictions
        lstm_prediction = target_scaler.inverse_transform(lstm_prediction_scaled).flatten()
        dense_prediction = target_scaler.inverse_transform(dense_prediction_scaled).flatten()

        # Calculate ensemble prediction
        ensemble_prediction = (lstm_prediction + dense_prediction) / 2

        return {
            'LSTM': float(lstm_prediction[0]),
            'Dense': float(dense_prediction[0]),
            'Ensemble': float(ensemble_prediction[0])
        }

    except Exception as e:
        return {'error': str(e)}

@app.route('/', methods=['GET'])
def home():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    try:
        # Get form data
        input_data = {
            'Temp': request.form['temperature'],
            'Rainfall': request.form['rainfall'],
            'Humidity': request.form['humidity'],
            'Demand': request.form['demand'],
            'Freshness': request.form['freshness'],
            'Inflation': request.form['inflation'],
            'Yield_per_Hectare': request.form['yield_per_hectare'],
            'Disaster': request.form['disaster'],
            'Season': request.form['season'],
            'Holiday_Festival': request.form['holiday_festival'],
            'Vegetable': request.form['vegetable']
        }

        # Make prediction
        predictions = make_prediction(input_data)

        if 'error' in predictions:
            return jsonify({'error': predictions['error']})

        return jsonify(predictions)

    except Exception as e:
        return jsonify({'error': str(e)})

if __name__ == '__main__':
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with stat
